# Prepare a verbal MWE lexicon

Create a JSON lexicon for `VerbalMultiWordExprPreTagger` from UTF-8 CSV files
with no header: one expression per row, one word per column.

In [1]:
import csv
import json
from pathlib import Path

from estnltk.vabamorf.morf import Vabamorf

## Input files and matching mode

Set the paths below, then run all cells. `True` matches non-anchor components
by lemma, allowing inflection; `False` matches their listed forms. The anchor
verb always matches by lemma.

In [2]:
sources = [
    (Path("väljendverb_käänduv.csv"), True),
    (Path("väljendverb_muutumatu.csv"), False),
]
output_path = Path("eki_verbal_mwe.json")

## Preparation

Vabamorf supplies possible lemmas without guessing or disambiguation. For these
lists, the anchor is the rightmost `ma`-infinitive, or otherwise the rightmost
verb, excluding `ei` and `ära`.

In [3]:
def analyse(form):
    return Vabamorf.instance().analyze(
        [form], disambiguate=False, guess=False, propername=True,
    )[0]['analysis']


def create_entry(forms, is_inflected, source):
    analyses = [analyse(form) for form in forms]
    verb_lemmas = [
        sorted({a['lemma'].casefold() for a in items
                if a['partofspeech'] == 'V' and a['lemma'] not in {'ei', 'ära'}})
        for items in analyses
    ]
    verbs = [i for i, lemmas in enumerate(verb_lemmas) if lemmas]
    infinitives = [i for i in verbs if forms[i].casefold() in verb_lemmas[i]]
    anchor = (infinitives or verbs)[-1]
    anchor_lemmas = [forms[anchor].casefold()] if infinitives else verb_lemmas[anchor]

    components = []
    for index, (form, items) in enumerate(zip(forms, analyses)):
        lemmas = sorted({a['lemma'].casefold() for a in items})
        components.append({
            'form': form,
            'lemmas': anchor_lemmas if index == anchor else lemmas,
            'match_by_lemma': is_inflected or index == anchor,
            'is_anchor': index == anchor,
        })
    return {
        'source': source,
        'expression': ' '.join(forms),
        'is_inflected': is_inflected,
        'components': components,
    }

## Examples

In [4]:
examples = [
    create_entry(["arveid", "õiendama"], True, "example"),
    create_entry(["aru", "saama"], False, "example"),
]
examples

[{'source': 'example',
  'expression': 'arveid õiendama',
  'is_inflected': True,
  'components': [{'form': 'arveid',
    'lemmas': ['arve'],
    'match_by_lemma': True,
    'is_anchor': False},
   {'form': 'õiendama',
    'lemmas': ['õiendama'],
    'match_by_lemma': True,
    'is_anchor': True}]},
 {'source': 'example',
  'expression': 'aru saama',
  'is_inflected': False,
  'components': [{'form': 'aru',
    'lemmas': ['aru'],
    'match_by_lemma': False,
    'is_anchor': False},
   {'form': 'saama',
    'lemmas': ['saama'],
    'match_by_lemma': True,
    'is_anchor': True}]}]

## Generate the JSON

In [5]:
entries = []
for path, is_inflected in sources:
    with path.open(encoding="utf-8", newline="") as source:
        entries.extend(create_entry(row, is_inflected, path.name) for row in csv.reader(source))

output_path.write_text(json.dumps(entries, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {len(entries)} expressions to {output_path}")

Wrote 2397 expressions to eki_verbal_mwe.json
